# SPE 10 Single Phase

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://drive.google.com/file/d/1X3UQadRKmtdI7MK6hP1HsHY2DJvO41g8/view?usp=drive_link&target=_blank)

*Author: Zakariya Abugrin | Date: Aug 2025*

In [1]:
# Colab only:
try:
    # Install reservoirflow in Colab
    import os
    import google.colab
    !pip install reservoirflow
    # Restart session after installation
    os.kill(os.getpid(), 9)
    print("Session was restarted.")
    print("Now you can run to the following cells.")
except ImportError:
    pass

## Introduction

This benchmark is used to evaluate the performance of `reservoirflow` in the most simple case which is single phase based on [SPE10](https://www.spe.org/web/csp/) with course grids based on [Benchmark Study](https://www.sintef.no/projectweb/geoscale/results/msmfem/spe10/) but for a single phase. This will be the basis for the two phase model.

![image](https://drive.google.com/thumbnail?id=11NhTbAU_lA768yiEAsoA18SshMjDtRqZ&sz=w1000)\
**the code used to generate this gif image is shown in [Save simulation run as gif](#save-simulation-run-as-gif) section below.*

## Import `reservoirflow`

We start with importing `reservoirflow` as `rf`. The abbreviation `rf` refers to `reservoirflow` where all modules under this library can be accessed. `rf` is also used throughout the [API](/api/API.html) documentation. We recommend our users to stick with this convention.

In [2]:
import reservoirflow as rf
import numpy as np

print(rf.__version__)

0.1.0b3


In [3]:
static = notebook = True

- Both `static` and `notebook` are set to `True` to produce images for this notebook. For an interactive visualization in your local machine, set both arguments to `False`. To allow for interactive visualizations in Colab, set only `static=False` and keep `notebook=True`. 

In [4]:
rf.utils.pyvista.set_mode("dark")

- By default, `"dark"` mode is used with a black background. You can change to `"light"` mode using `rf.utils.pyvista.set_mode()` function. 

## Define a regular cartesian grid

We start with defining the grid model which represents the rock geometry and properties using `grids` module. Currently, only `RegularCartesian` grid models are supported. In the near future, `Radial` and `IrregularCartesian` models will be added.

In [30]:
grid = rf.grids.RegularCartesian(
    nx=9 , # 10, #
    ny=11, # 22, #
    nz=5, # 5, #
    dx=133.3, # 120, #
    dy=200, # 100, #
    dz=34, # 34, #
    kx=1,
    ky=1,
    kz=1,
    phi=0.30,
    z=12000, #
    comp=10e-6, #
    unit="field",
)

```{note}
Note that `field` units are used in this tutorial. For more information about selecting a unit system or what units or factors are used for each property, see [Units & Factors](/user_guide/units_factors/units_factors.html).
```

The grid object can be shown using `show()` method. Argument `label` can be used to show cells id or other properties. For more information, check [`RegularCartesian`](/api/reservoirflow.grids.RegularCartesian.html).

In [31]:
grid.show(
    label='id',
    boundary=False,
    static=static,
    notebook=notebook,
)

```{note}
Cells are given an id based on natural order starting form the bottom left corner, see `grid.cells_id`. Cells id increase in x from left to right, in y from front to back, and in z from bottom to top. Boundaries are also counted. You can use the cell id to access its properties, or add a well in that location as shown in [Add wells](#add-wells) section below.
```

## Define a single phase compressible fluid

After defining the grid model, we can use the `fluids` module to define a fluid. A single phase fluid can be defined using `SinglePhase` class which accepts arguments such s viscosity (mu), formation volume factor (B), density (rho), compressibility (comp) etc. For more information, check [`SinglePhase`](/api/reservoirflow.fluids.SinglePhase.html).

In [32]:
fluid = rf.fluids.SinglePhase(
    mu=0.3, # water
    B=1.01, # water
    rho=64, # water at surface
    comp=3.10e-6, # water
    unit="field",
)

## Create a reservoir simulation model

To construct a reservoir simulation model, both `fluid` and `grid` objects are required as input arguments in `models` module. In addition, you can also define the initial reservoir pressure (pi), time step duration (dt), start date etc. For more information, check [`models`](/api/reservoirflow.models.html). Currently, only `BlackOil` models are supported. In the future, more advanced models will be added such as `Compositional` or `Thermal`.

In [33]:
model = rf.models.BlackOil(
    grid=grid,
    fluid=fluid,
    pi=6000, #
    dt=1,
    start_date="01.01.2000",
    unit="field",
)

## Define boundary conditions

If you start the simulation run without defining a boundary condition, then all boundaries are set to zero flow rate. To define a boundary condition, you can use `set_boundaries()` method where you specify the boundary condition as dictionary where keys are cells id and values are tuple of ("cond", value). For example, a constant zero rate boundary condition in `cell_id=0` can be set as `model.set_boundaries({0: ("rate", 0)})`. 

Below we set all the boundaries cells to zero rate (default behavior).

In [34]:
model.set_boundaries({i: ("rate", 0) for i in grid.get_boundaries("id", fmt="array")})

## Add wells

To make a five spot pattern, single cell wells are used to add wells to specific cells as following:

### Injection wells

- 4 injectors: 182, 186, 218, 222

In [35]:
# injection_wells = {
#     'inj1': [1577, 1001],
#     'inj2': [1589, 1002],
#     'inj3': [0, 1013],
#     'inj4': [0, 1014],
# }

# for cell_id in [1001, 1002, 1013, 1014]:
for cell_id in [500,]:
    model.set_well(cell_id=cell_id, 
                #    q=5000, # pressure will exceed 10000 
                   pwf=10000, #
                   s=0,
                   r=3.5
                   )

```{note}
Since the bottom hole flowing pressure (BHFP or `pwf`) is higher than the initial reservoir pressure, then these wells will have an injection rates (i.e. a positive flow rate means additional fluid is injected into the reservoir). This might not be the expected behavior in reservoir simulation and this might be reversed in the future where injection rates will be expressed as a negative rate (depends on the users' feedback). Feel free to comment on this.
```

### Production well

- 1 producer: 202

In [36]:
# production_wells = {
#     'prod1': [301, 589, 877, 1165, 1453],
#     'prod2': [310, 598, 886, 1174, 1462],
#     'prod3': [562, 850, 1138, 1426, 1714],
#     'prod4': [553, 841, 1129, 1417, 1705],
# }

# for cell_id in [877, 886, 1138, 1129]:
for cell_id in [441, 449, 559, 551]:

    model.set_well(
        cell_id=cell_id,
        q=-4000,
        pwf=1000,
        s=0,
        r=3.5,
    )

```{note}
In this case, `pwf` is lower than the initial reservoir pressure, which will lead to a production rates (i.e. a negative flow rate means a fluid is produced from the reservoir).
```

## Compile the model

Before you can run the model, you need to compile a solution for it. By compiling a solution, you actually decide the solution you want to use for your model. Interestingly, `reservoirflow` provides multiple solutions for the same model based on your configuration. 

```{hint}
Compiling solutions is the most interesting idea introduced in ``reservoirflow`` which allows to solve the same model using different solutions so we can compare them with each other and/or combine them together.
```

Currently, a `numerical` solution based on Finite-Difference-Method (`FDM`) is available. You can read more about the available solution in the [`solutions`](/api/reservoirflow.solutions.html). Below, we compile our model using a `numerical` solution using `FDM`.

In [81]:
model.compile(stype="numerical", method="FDM", sparse=True)

[info] FDM was assigned as model.solution.


```{attention}
A model can not be run until it is complied. Methods such as ``solve()`` and ``run()`` will be functioning only after the model is compiled.
```

## Run the model with 40 time steps

To run the simulation model, use `run()` method as shown below. Here, you can set the number of time steps based on `nsteps` argument. Note that each time step will have a time duration as defined by `dt` argument when the model was constructed. 

There are two modes available run the model based on `vectorize` argument, `True` means vectorized and `False` means symbolized. In addition, you can also select a solver which can be `direct`, `iterative`, or `neurical`. Developing solvers based on neural-networks is also a new idea introduced by ``reservoirflow``.

<!-- ```{tip}
- Use ``vectorize=True`` mode for better performance especially when you have a large model. 
- Use ``vectorize`` mode only to see how the system of linear equations is built which might be very useful for small models and to verify the `vectorized` implementation. 
- Use ``direct`` for a lower computing errors as long as ``iterative`` does not offer any additional performance boost. For now, ``neurical`` solvers remains one of our research topics.
``` -->

In [ ]:
model.run(
    nsteps=2000, #
    vectorize=True,
    isolver=None,
)

[info] Simulation run started: 2000 timesteps.


[step]: 100%|██████████| 2000/2000 [21:04<00:00,  1.58steps/s]  

[info] Simulation run of 2000 steps finished in 1264.85 seconds.
[info] Material Balance Error: 9.063285294774737e-05.


## Visualize the simulation run in 3D show

You can interactively show the simulation run in a 3D show using `show()` method. There is a lot of functionality here. For interactive visualization, set `static` and `notebook` to `False`. For more information, check the [`BlackOil.show()`](/api/reservoirflow.models.BlackOil.show.html#reservoirflow.models.BlackOil.show). 

```{note}
Here, boundaries are also added (`boundary=True`) but they appear in a gray color. That is because boundaries are defined as a zero flow rate where pressures are undefined (i.e. `np.nan`). If boundary conditions were set to constant pressures, values will appear based on the defined color map.
```

In [84]:
static = notebook = False
model.show(
    prop="pressures",
    boundary=False,
    cmap="Blues",
    static=static,
    notebook=notebook,
)

```{tip}
Use the color maps names as `str` with `cmap` argument. Matplotlib color maps are supported, for more color maps, check [Choosing Colormaps in Matplotlib](https://matplotlib.org/stable/users/explain/colors/colormaps.html). In the above image, `cmap="Blues"` was used. In [Save simulation run as gif](#save-simulation-run-as-gif) section below, `cmap="tab10"` was used instead. 
```

## Show results as a pandas `DataFrame`

The results of the Simulation run can be accessed as a pandas `DateFrame` using `get_df()` method. You can select `columns` and `scale` values. For more information, check [`BlackOil.get_df()`](/api/reservoirflow.models.BlackOil.get_df.html#reservoirflow.models.BlackOil.get_df).

In [80]:
model.update_scalers(True)
property = "wells_pressure"
filepath = f"data_4_1/model_spe10_4_1_single_phase_{property}_scale.csv"
df = model.get_df(columns=["time", "date", property], 
                  melt=False, 
                  scale=True, 
                  units=True,
                  )
df.to_csv(filepath)


df

,Time [scaled],Date [d.m.y],Pwf500 [scaled],Pwf441 [scaled],Pwf449 [scaled],Pwf559 [scaled],Pwf551 [scaled]
Step,,,,,,,
0,0.0000,01.01.2000,0.764027,0.764027,0.764027,0.764027,0.764027
1,0.0005,02.01.2000,2.800686,-1.781796,-1.781796,-1.781796,-1.781796
2,0.0010,03.01.2000,2.800686,-1.781796,-1.781796,-1.781796,-1.781796
3,0.0015,04.01.2000,2.800686,-1.781796,-1.781796,-1.781796,-1.781796
4,0.0020,05.01.2000,2.800686,-1.781796,-1.781796,-1.781796,-1.781796
...,...,...,...,...,...,...,...
1996,0.9980,19.06.2005,2.800686,-1.781796,-1.781796,-1.781796,-1.781796
1997,0.9985,20.06.2005,2.800686,-1.781796,-1.781796,-1.781796,-1.781796
1998,0.9990,21.06.2005,2.800686,-1.781796,-1.781796,-1.781796,-1.781796


In [54]:
import pandas as pd

pd.read_csv("data_4_4/model_spe10_4_4_single_phase_wells_pressure_melt.csv", index_col=0)

,id,Step,x,y,z,Time [days],Date [d.m.y]
0,0,0,60.0,50.0,17.0,0,01.01.2000
1,1,0,180.0,50.0,17.0,0,01.01.2000
2,2,0,300.0,50.0,17.0,0,01.01.2000
3,3,0,420.0,50.0,17.0,0,01.01.2000
4,4,0,540.0,50.0,17.0,0,01.01.2000
...,...,...,...,...,...,...,...
4034011,2011,2000,900.0,2350.0,221.0,2000,23.06.2005
4034012,2012,2000,1020.0,2350.0,221.0,2000,23.06.2005
4034013,2013,2000,1140.0,2350.0,221.0,2000,23.06.2005
4034014,2014,2000,1260.0,2350.0,221.0,2000,23.06.2005


In [ ]:
import pickle

# Store data:
file_name = "model_spe10_4_4_single_phase.pkl"
with open(file_name, "wb") as file:
    pickle.dump(model, file)

with open(file_name, "rb") as file:
    model_loaded = pickle.load(file)

## Save simulation run as gif

You can save gif movie of the simulation run using `save_gif()` method. For more information, check the [`BlackOil.save_gif()`](/api/reservoirflow.models.BlackOil.save_gif.html#reservoirflow.models.BlackOil.save_gif).

In [ ]:
if not static:
    model.save_gif(
        prop="pressures",
        boundary=False,
        cmap="tab10",
        file_name="attachments/grid_animated.gif",
        window_size=rf.utils.pyvista.get_window_size("hd"),
    )

```{tip}
To save a gif movie of the simulation run, set `static=False` or just remove the first line and unindent.
```

```{include} /_static/comments_section.md
```